# Camera-Based Grasp Detection with WGAN
This notebook covers:
1. Training WGAN with the same architecture
2. Exporting weights to Q6.10 `.mem`
3. Running fixed-point inference with camera

## 1. Install Dependencies

In [1]:
!pip install torch torchvision opencv-python numpy matplotlib

## 2. Imports & Config

In [2]:

import os, copy, cv2, numpy as np, torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader


## 3. Tahap Preprocessing

In [17]:
import os
import cv2
import numpy as np
from PIL import Image

SRC_ROOT = "cornell_grasp"          # folder dataset Cornell
DST_FOLDER = "cornell_grasp_bmp"    # output bitmap 28x28
os.makedirs(DST_FOLDER, exist_ok=True)

def parse_grasp_file(txt_file):
    grasps = []
    with open(txt_file, "r") as f:
        lines = f.readlines()

        for i in range(0, len(lines), 4):
            pts = []
            valid = True

            for j in range(4):
                parts = lines[i+j].strip().split()
                if len(parts) != 2:
                    valid = False
                    break

                x = float(parts[0])
                y = float(parts[1])

                if np.isnan(x) or np.isnan(y):
                    valid = False
                    break

                pts.append((int(x), int(y)))

            if valid:
                grasps.append(pts)

    return grasps

count = 0
for root, dirs, files in os.walk(SRC_ROOT):
    for file in files:
        if file.endswith("cpos.txt"):   # hanya grasp sukses

            base = file.replace("cpos.txt", "")
            img_file = os.path.join(root, base + "r.png")   # sesuai dataset kamu
            txt_file = os.path.join(root, file)

            if not os.path.exists(img_file):
                print("Image not found:", img_file)
                continue

            img = cv2.imread(img_file, cv2.IMREAD_GRAYSCALE)
            h, w = img.shape

            grasps = parse_grasp_file(txt_file)

            for g in grasps:
                mask = np.zeros((h, w), dtype=np.uint8)
                pts = np.array(g, np.int32)
                cv2.fillConvexPoly(mask, pts, 255)

                mask28 = cv2.resize(mask, (28, 28))
                mask28 = mask28.astype(np.uint8)

                out = Image.fromarray(mask28)
                out.save(os.path.join(DST_FOLDER, f"grasp_{count:06d}.png"))
                count += 1

print("Total grasp bitmap:", count)


Total grasp bitmap: 5110


## 4. Define WGAN Architecture

In [20]:
import os
import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader

def load_bitmap_folder(folder):
    files = sorted([f for f in os.listdir(folder) if f.endswith(".png") or f.endswith(".bmp")])
    data = []
    for f in files:
        img = Image.open(os.path.join(folder, f)).convert("L").resize((28,28))
        arr = np.array(img, dtype=np.float32) / 255.0
        arr = arr * 2 - 1   # [-1, 1]
        data.append(arr.flatten())
    return np.array(data)

# GANTI dengan folder hasil preprocessing Cornell kamu
dataset_np = load_bitmap_folder("cornell_grasp_bmp")

dataset = torch.tensor(dataset_np, dtype=torch.float32)
loader = DataLoader(dataset, batch_size=128, shuffle=True, drop_last=True)

print("Dataset shape:", dataset.shape)


Dataset shape: torch.Size([5110, 784])


In [21]:

LATENT = 64
device = "cuda" if torch.cuda.is_available() else "cpu"

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(LATENT, 256)
        self.l2 = nn.Linear(256, 256)
        self.l3 = nn.Linear(256, 784)
        self.act = nn.ReLU()
        self.tanh = nn.Tanh()
    def forward(self, z):
        return self.tanh(self.l3(self.act(self.l2(self.act(self.l1(z))))))

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(784,256)
        self.l2 = nn.Linear(256,256)
        self.l3 = nn.Linear(256,256)
        self.l4 = nn.Linear(256,1)
        self.leaky = nn.LeakyReLU(0.2)
    def forward(self, x):
        x = self.leaky(self.l1(x))
        x = self.leaky(self.l2(x))
        x = self.leaky(self.l3(x))
        return self.l4(x)

G = Generator().to(device)
C = Critic().to(device)


## 5. Training Loop

In [23]:

def lipschitz_penalty(C, real, fake, lambda_lp=5):
    alpha = torch.rand(real.size(0),1).expand_as(real).to(real.device)
    interpolated = (alpha*real + (1-alpha)*fake).detach().requires_grad_(True)
    score = C(interpolated)
    grads = torch.autograd.grad(score, interpolated, torch.ones_like(score), create_graph=True)[0]
    grad_norm = grads.view(grads.size(0),-1).norm(2,dim=1)
    return lambda_lp * ((grad_norm-1).clamp(min=0)**2).mean()

optG = optim.Adam(G.parameters(), lr=5e-5, betas=(0.5,0.9))
optC = optim.Adam(C.parameters(), lr=5e-5, betas=(0.5,0.9))

for epoch in range(200):
    for real in loader:
        real = real.to(device)
        b = real.size(0)
        for _ in range(5):
            z = torch.randn(b, LATENT).to(device)
            fake = G(z).detach()
            lossC = -(C(real).mean() - C(fake).mean()) + lipschitz_penalty(C, real, fake)
            optC.zero_grad(); lossC.backward(); optC.step()

        z = torch.randn(b, LATENT).to(device)
        fake = G(z)
        lossG = -C(fake).mean()
        optG.zero_grad(); lossG.backward(); optG.step()
    print("Epoch", epoch, "C:", lossC.item(), "G:", lossG.item())


Epoch 0 C: -1.4351941347122192 G: -23.04848289489746
Epoch 1 C: -0.47857606410980225 G: -20.740699768066406
Epoch 2 C: -0.14510123431682587 G: -19.189056396484375
Epoch 3 C: -0.042771365493535995 G: -17.944612503051758
Epoch 4 C: -0.019528793171048164 G: -18.506988525390625
Epoch 5 C: -0.0013287063920870423 G: -19.02250099182129
Epoch 6 C: -0.0016994476318359375 G: -19.21778678894043
Epoch 7 C: -0.0046901702880859375 G: -17.6490478515625
Epoch 8 C: 0.002712249755859375 G: -16.30986213684082
Epoch 9 C: -0.021383285522460938 G: -19.45801544189453
Epoch 10 C: -0.0032625198364257812 G: -13.15832233428955
Epoch 11 C: -0.0002384185791015625 G: -11.621349334716797
Epoch 12 C: 0.005084037780761719 G: -13.191089630126953
Epoch 13 C: -0.0027751922607421875 G: -16.289165496826172
Epoch 14 C: -0.018472671508789062 G: -14.026190757751465
Epoch 15 C: 0.003437042236328125 G: -11.016416549682617
Epoch 16 C: 0.0013866424560546875 G: -8.923771858215332
Epoch 17 C: -0.000873565673828125 G: -8.13493156433

## 6. Export Weights to Q6.10

In [25]:
import os

def float_to_q6_10_hex(v):
    v = max(-1.0, min(1.0, float(v)))
    scaled = int(round(v * 1024))
    return f"{scaled & 0xFFFF:04X}"

def export_layer(layer, name):
    os.makedirs("mem_weights", exist_ok=True)
    W = layer.weight.data.cpu().numpy().flatten()
    B = layer.bias.data.cpu().numpy().flatten()

    with open(f"mem_weights/{name}_W.mem", "w") as fw:
        for v in W:
            fw.write(float_to_q6_10_hex(v) + "\n")

    with open(f"mem_weights/{name}_B.mem", "w") as fb:
        for v in B:
            fb.write(float_to_q6_10_hex(v) + "\n")

# Export semua layer Critic
export_layer(C.l1, "C_l1")
export_layer(C.l2, "C_l2")
export_layer(C.l3, "C_l3")
export_layer(C.l4, "C_l4")

print("Weights exported to mem_weights/")


Weights exported to mem_weights/


## 7. Fixed-Point Inference with Camera

In [32]:

Q=10
def q610_to_float(h):
    v=int(h,16)
    if v&0x8000: v=-((~v & 0xFFFF)+1)
    return v/(1<<Q)

def load_mem(f): return np.array([q610_to_float(l.strip()) for l in open(f)])

C1_W = load_mem("mem_weights/C_l1_W.mem").reshape(256,784)
C1_B = load_mem("mem_weights/C_l1_B.mem")
C2_W = load_mem("mem_weights/C_l2_W.mem").reshape(256,256)
C2_B = load_mem("mem_weights/C_l2_B.mem")
C3_W = load_mem("mem_weights/C_l3_W.mem").reshape(256,256)
C3_B = load_mem("mem_weights/C_l3_B.mem")
C4_W = load_mem("mem_weights/C_l4_W.mem").reshape(1,256)
C4_B = load_mem("mem_weights/C_l4_B.mem")

def leaky(x,a=0.2): return np.where(x>0,x,a*x)
def critic_forward(x):
    x = leaky(C1_W@x + C1_B)
    x = leaky(C2_W@x + C2_B)
    x = leaky(C3_W@x + C3_B)
    return (C4_W@x + C4_B)[0]

cap=cv2.VideoCapture(1)
while True:
    ret,frame=cap.read()
    if not ret: break
    gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    crop=cv2.resize(gray,(28,28))
    x=crop.astype(np.float32)/255.0
    x=x*2-1
    score=critic_forward(x.flatten())
    cv2.putText(frame,f"Grasp Score: {score:.3f}",(20,40),cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)
    cv2.imshow("Grasp Detection",frame)
    if cv2.waitKey(1)&0xFF==27: break
cap.release(); cv2.destroyAllWindows()
